# Build scenario-level dataset for scenario discovery

Reads the Scenario Compass Initiative pathways ensemble
(`data/SCI-2025_v1.0_pathways_ensemble_global.xlsx`) and produces one clean, scenario-level table
(one row per Model-Scenario) holding exactly what notebooks 02 and 03 need:

- the two `meta`-sheet fields that define the outcome (year of net-zero CO2, cumulative CCS)
- the **seven** pathway-descriptor factors we analyse, pivoted out of the long `data` sheet as full
  2010-2100 trajectories, so any snapshot year - we use **2030** and **2050** - is available
- the binary outcome labels (`success_nz2070`, `low_ccs_reliance`, `desired_success`)

Output: `data_for_scenariodiscovery_full.csv`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)

In [2]:
XLSX_PATH = "../../data/SCI-2025_v1.0_pathways_ensemble_global.xlsx"
ENGINE = "calamine"  # python-calamine: much faster than openpyxl on this 120MB file
OUTPUT_CSV = "data_for_scenariodiscovery_full.csv"

## 1. Load the `meta` sheet (outcome fields)

One row per Model-Scenario. We only need two fields from it, both used to build the outcome:
`Emissions Diagnostics|Year of Net Zero|CO2` (net-zero timing) and
`Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]` (CCS reliance).

In [3]:
meta = pd.read_excel(XLSX_PATH, sheet_name="meta", engine=ENGINE)
print(meta.shape)

NZ_COL = "Emissions Diagnostics|Year of Net Zero|CO2"
CCS_COL = "Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]"
meta[[NZ_COL, CCS_COL]].describe()

(1599, 71)


,Emissions Diagnostics|Year of Net Zero|CO2,"Emissions Diagnostics|Cumulative CCS [2020-2100, Gt CO2]"
count,909.000000,1509.000000
mean,2070.380638,522.127105
std,14.398448,399.460534
min,2030.000000,0.000000
25%,2060.000000,219.651632
50%,2070.000000,493.199712
75%,2080.000000,729.805161
max,2100.000000,2145.347648


## 2. Load the `data` sheet and pivot the seven factors to wide format

The `data` sheet is long IAMC format (Model, Scenario, Region, Variable, Unit, one column per year);
Region is always `World`. We keep the full trajectory for exactly the seven variables analysed in
notebook 03 - including the two outputs added to this factor set,
`Carbon Capture|Geological Storage|Biomass` (BECCS) and the MAGICC median GSAT temperature - then
pivot `Variable` into columns suffixed by year (`<variable>|<year>`).

In [ ]:
FACTORS = [
    "Emissions|CO2",
    "Primary Energy|Fossil",
    "Primary Energy|Biomass",
    "Primary Energy",
    "Primary Energy|Non-Biomass Renewables",
    "Carbon Capture|Geological Storage",
    "Final Energy|Electricity",
    "Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]",
]

data_long = pd.read_excel(XLSX_PATH, sheet_name="data", engine=ENGINE)
assert data_long["Region"].unique().tolist() == ["World"]

missing = set(FACTORS) - set(data_long["Variable"].unique())
assert not missing, f"Variables not found in data sheet: {missing}"

filtered = data_long[data_long["Variable"].isin(FACTORS)].copy()
year_cols = [c for c in filtered.columns if c.isdigit()]

wide = filtered.set_index(["Model", "Scenario", "Variable"])[year_cols].unstack("Variable")
wide.columns = [f"{var}|{year}" for year, var in wide.columns]
wide = wide.reset_index()
print(wide.shape)
wide.head()

(1591, 135)


,Model,Scenario,Carbon Capture|Geological Storage|2010,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2010,Emissions|CO2|2010,Final Energy|Electricity|2010,Primary Energy|Biomass|2010,Primary Energy|Fossil|2010,Primary Energy|Non-Biomass Renewables|2010,Carbon Capture|Geological Storage|2015,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2015,Emissions|CO2|2015,Final Energy|Electricity|2015,Primary Energy|Biomass|2015,Primary Energy|Fossil|2015,Primary Energy|Non-Biomass Renewables|2015,Carbon Capture|Geological Storage|2020,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2020,Emissions|CO2|2020,Final Energy|Electricity|2020,Primary Energy|Biomass|2020,Primary Energy|Fossil|2020,Primary Energy|Non-Biomass Renewables|2020,Carbon Capture|Geological Storage|2025,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2025,Emissions|CO2|2025,Final Energy|Electricity|2025,Primary Energy|Biomass|2025,Primary Energy|Fossil|2025,Primary Energy|Non-Biomass Renewables|2025,...,Primary Energy|Fossil|2080,Primary Energy|Non-Biomass Renewables|2080,Carbon Capture|Geological Storage|2085,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2085,Emissions|CO2|2085,Final Energy|Electricity|2085,Primary Energy|Biomass|2085,Primary Energy|Fossil|2085,Primary Energy|Non-Biomass Renewables|2085,Carbon Capture|Geological Storage|2090,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2090,Emissions|CO2|2090,Final Energy|Electricity|2090,Primary Energy|Biomass|2090,Primary Energy|Fossil|2090,Primary Energy|Non-Biomass Renewables|2090,Carbon Capture|Geological Storage|2095,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2095,Emissions|CO2|2095,Final Energy|Electricity|2095,Primary Energy|Biomass|2095,Primary Energy|Fossil|2095,Primary Energy|Non-Biomass Renewables|2095,Carbon Capture|Geological Storage|2100,Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7.5.3]|2100,Emissions|CO2|2100,Final Energy|Electricity|2100,Primary Energy|Biomass|2100,Primary Energy|Fossil|2100,Primary Energy|Non-Biomass Renewables|2100
0,AIM/CGE 2.0,SSP1-19,0.0,0.975563,35782.6485,64.5446,44.9428,400.6963,13.3812,0.0,1.115697,36508.26915,74.22065,43.70495,418.17680,17.40440,0.0,1.225605,37233.8898,83.8967,42.4671,435.6573,21.4276,235.6808,1.381006,28145.69155,97.36630,43.60175,357.45595,48.88150,...,57.8612,243.9796,NaN,1.205440,NaN,NaN,NaN,NaN,NaN,5405.1347,1.177504,-4390.4928,248.5773,86.6641,54.7800,240.2778,NaN,1.162928,NaN,NaN,NaN,NaN,NaN,5253.5049,1.132365,-4474.8741,243.8105,86.7606,46.9562,236.0284
1,AIM/CGE 2.0,SSP1-26,0.0,0.975563,35781.8143,64.5452,44.9429,400.7582,13.3809,0.0,1.115698,36481.57640,74.23840,43.70375,418.27880,17.40285,0.0,1.225277,37181.3385,83.9316,42.4646,435.7994,21.4248,0.0000,1.378572,33485.92355,95.00225,41.72640,419.22350,33.28940,...,192.0237,183.2110,NaN,1.648127,NaN,NaN,NaN,NaN,NaN,9117.6701,1.629364,626.9675,225.3647,76.8439,171.6095,185.2776,NaN,1.631904,NaN,NaN,NaN,NaN,NaN,9219.4939,1.609574,17.7377,222.4464,77.9287,164.1606,180.6178
2,AIM/CGE 2.0,SSP1-34,0.0,0.975563,35781.8143,64.5452,44.9429,400.7582,13.3809,0.0,1.115698,36481.57640,74.23840,43.70375,418.27880,17.40285,0.0,1.225268,37181.3385,83.9316,42.4646,435.7994,21.4248,0.0000,1.377366,36191.56205,96.67850,41.38820,441.53905,30.42620,...,260.9164,147.3679,NaN,2.065856,NaN,NaN,NaN,NaN,NaN,5525.6711,2.071235,10882.1926,211.4716,85.0360,245.1817,151.4369,NaN,2.089055,NaN,NaN,NaN,NaN,NaN,6312.4080,2.090244,9037.7818,212.5913,87.8550,230.8559,151.7119
3,AIM/CGE 2.0,SSP1-45,0.0,0.975563,35781.8143,64.5452,44.9429,400.7582,13.3809,0.0,1.115711,36185.22300,74.17075,43.76595,415.62180,17.78775,0.0,1.226074,36588.6317,83.7963,42.5890,430.4854,22.1946,0.0000,1.377311,38023.76690,97.55580,41.42160,447.39335,30.21680,...,344.0632,121.5186,NaN,2.597730,NaN,NaN,NaN,NaN,NaN,0.0000,2.620006,21544.6626,206.3124,73.9360,307.2986,129.4012,NaN,2.652736,NaN,NaN,N

## 3. Merge meta + pivoted factors

Left-join on `meta` (the master 1599-row Model-Scenario list) so a scenario missing one factor still
appears, just with NaNs for that column.

In [5]:
merged = meta.merge(wide, on=["Model", "Scenario"], how="left", validate="one_to_one")
print(merged.shape)

(1599, 204)


## 4. Build the outcome: net zero by 2070 *and* low CCS reliance

- `success_nz2070`: net-zero CO2 reached at or before 2070 (NaN year = never within horizon -> failure)
- `low_ccs_reliance`: cumulative CCS (2020-2100) at or below a fixed 1000 Gt CO2 threshold
  (~the ensemble median); scenarios with no reported cumulative CCS can't be confirmed low-reliance
  and are excluded from the desired region rather than counted as low
- `desired_success`: both - reaches net zero early *without* leaning heavily on CCS

In [6]:
CCS_THRESHOLD = 1000  # Gt CO2, fixed cut (~ensemble median of cumulative CCS)

merged["success_nz2070"] = merged[NZ_COL].le(2070)
merged["low_ccs_reliance"] = merged[CCS_COL] <= CCS_THRESHOLD
merged["desired_success"] = merged["success_nz2070"] & merged["low_ccs_reliance"]

print(f"Missing cumulative CCS: {merged[CCS_COL].isna().sum()} scenarios")
print(pd.crosstab(merged["success_nz2070"], merged["low_ccs_reliance"], dropna=False, margins=True))
print(f"\ndesired_success (NZ<=2070 AND low CCS): {merged['desired_success'].sum()} scenarios "
      f"({merged['desired_success'].mean():.1%} of ensemble)")

Missing cumulative CCS: 90 scenarios
low_ccs_reliance  False  True   All
success_nz2070                     
False               147   955  1102
True                125   372   497
All                 272  1327  1599

desired_success (NZ<=2070 AND low CCS): 372 scenarios (23.3% of ensemble)


## 5. Sanity check: factor coverage at the 2030 and 2050 snapshots

Notebook 03 runs PRIM/CART on the 2030 and on the 2050 value of each factor (intersection of
non-missing values per year), so it's worth seeing how many scenarios survive `dropna` at each year.

In [7]:
for year in [2040, 2050]:
    cols = [f"{v}|{year}" for v in FACTORS]
    complete = merged[cols].dropna()
    print(f"--- {year}: {len(complete)} scenarios complete on all {len(FACTORS)} factors ---")
    for v in FACTORS:
        col = f"{v}|{year}"
        print(f"  {col[:62]:62s} missing={merged[col].isna().mean():5.1%}")
    print()

--- 2040: 1355 scenarios complete on all 7 factors ---
  Emissions|CO2|2040                                             missing= 0.9%
  Primary Energy|Fossil|2040                                     missing= 1.5%
  Primary Energy|Biomass|2040                                    missing= 1.3%
  Primary Energy|Non-Biomass Renewables|2040                     missing= 8.7%
  Carbon Capture|Geological Storage|2040                         missing= 5.4%
  Final Energy|Electricity|2040                                  missing= 2.6%
  Climate Assessment|Surface Temperature (GSAT)|Median [MAGICCv7 missing= 3.6%

--- 2050: 1355 scenarios complete on all 7 factors ---
  Emissions|CO2|2050                                             missing= 0.9%
  Primary Energy|Fossil|2050                                     missing= 1.5%
  Primary Energy|Biomass|2050                                    missing= 1.3%
  Primary Energy|Non-Biomass Renewables|2050                     missing= 8.7%
  Carbon Capture|Geo

## 6. Save

In [8]:
merged.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {merged.shape} to {OUTPUT_CSV}")

Saved (1599, 207) to data_for_scenariodiscovery_full.csv
